# System Metrics: what training *cost*, and where the bottleneck is

This is an **advanced** tutorial. It assumes the tracking notebooks (`b_`–`e_`).

Every metric we've logged so far answers *"is the model any good?"* — RMSE, R², ROC-AUC. **System metrics** answer a different question that production ML teams care about just as much: *"what did this training run **cost**, and is the hardware the bottleneck?"* CPU saturation, RAM headroom, GPU utilization, disk and network I/O — the **resource-observability** half of MLOps.

Why log them next to the accuracy numbers?

- **Right-sizing & cost.** A run that pegs one CPU core at 100% while 21 sit idle is telling you the job isn't parallelized; a GPU job that never lifts GPU utilization above 5% is paying for a GPU it isn't using. On cloud hardware that's money.
- **Bottleneck diagnosis.** Is the slow epoch CPU-bound, memory-bound, or starved on disk I/O? The time series tells you without guessing.
- **Reproducibility of *performance*, not just results.** "It trained in 4 minutes on this box" is part of the run's record, comparable across runs and machines.

This is a **standalone** notebook; the capstone (`k_`) flips system-metrics logging on for its heavy tuning step and cross-links back here.

## Source material

- **[System metrics docs](https://mlflow.org/docs/latest/ml/tracking/system-metrics/)** — the `system/*` namespace, the enable flags, and the sampling knobs.
- **[XGBoost GPU support](https://xgboost.readthedocs.io/en/stable/gpu/index.html)** — `device="cuda"`, `tree_method="hist"`.

## What MLflow logs, and how to turn it on

Three ways to enable it; the metrics are identical:

| How | Scope |
|---|---|
| `mlflow.start_run(log_system_metrics=True)` | one run |
| `mlflow.enable_system_metrics_logging()` | every run in the process |
| env var `MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING=true` | every run, no code change |

A background thread samples the host and logs to the **`system/`** metric namespace, so the values render as **time series** in the run UI (not single numbers). Two knobs control cadence:

- `mlflow.set_system_metrics_sampling_interval(seconds)` — how often to sample (default **10 s**).
- `mlflow.set_system_metrics_samples_before_logging(n)` — how many samples to average before writing a point (default **1**).

For a short training run, 10 s is far too coarse — you'd get one or two points. We drop it to **1 s** below so the charts are legible.

**Dependencies.** CPU / RAM / disk / network come from **`psutil`**, which ships with MLflow. **GPU metrics need the NVIDIA bindings** — see the prerequisites note: install **`nvidia-ml-py`**. Without it, MLflow silently logs everything *except* the `system/gpu_*` series.

## Prerequisites

### The GPU dependency — `nvidia-ml-py`, not `pynvml`

MLflow reads GPU stats through the `pynvml` **module**. There are two PyPI packages that provide it, and this is a live version-drift trap:

- **`nvidia-ml-py`** — NVIDIA's official bindings. **Use this one.**
- **`pynvml`** — an older third-party wrapper, now **deprecated**; importing it prints a `FutureWarning` telling you to install `nvidia-ml-py` instead.

So add the official package (a reader following an older blog post will likely see `uv add pynvml` — that's the stale advice):

```bash
uv add nvidia-ml-py
```

(`pip` is forbidden in this repo — see `CLAUDE.md`. Commit `pyproject.toml` and `uv.lock` together.) `xgboost` is already a project dependency.

### This machine (where the charts came from)

The outputs in this notebook were produced on a laptop with **22 CPU cores, 62 GB RAM, and an NVIDIA RTX 2000 Ada Generation Laptop GPU (8 GB VRAM)**. The GPU panel is real here — *but only because the GPU run below actually uses CUDA*. Your numbers will differ; the **shapes** are the lesson.

### Start the tracking server

In a separate terminal from the repo root (same as `f_`/`g_`/`h_`):

```bash
mlflow ui --host 127.0.0.1 --port 5001
```

In [1]:
import time

import numpy as np
import xgboost as xgb
from sklearn.datasets import make_regression

import mlflow
from mlflow.exceptions import RestException
from mlflow.tracking import MlflowClient

HOST, PORT = "127.0.0.1", 5001
mlflow.set_tracking_uri(f"http://{HOST}:{PORT}")

# Sample every second and log every sample, so short runs still produce a
# readable curve (defaults: 10 s interval, 1 sample-before-logging).
mlflow.set_system_metrics_sampling_interval(1)
mlflow.set_system_metrics_samples_before_logging(1)

experiment_name = "System Metrics"
try:
    mlflow.create_experiment(name=experiment_name)
except RestException as e:
    if "RESOURCE_ALREADY_EXISTS" not in str(e):
        raise
mlflow.set_experiment(experiment_name)

client = MlflowClient()

## Step 1: Turn it on and see the namespace

One run with `log_system_metrics=True`. We do a little work and sleep a few seconds so the sampler captures several points, then list every `system/*` key that landed on the run.

In [2]:
with mlflow.start_run(run_name="hello-system-metrics", log_system_metrics=True) as run0:
    # trivial busy-work so the sampler has something non-zero to record
    _ = sum(i * i for i in range(5_000_000))
    time.sleep(5)
    run0_id = run0.info.run_id

system_keys = sorted(k for k in client.get_run(run0_id).data.metrics if k.startswith("system/"))
print(f"{len(system_keys)} system/* metrics logged:\n")
for k in system_keys:
    print("  ", k)

2026/06/02 14:22:34 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


2026/06/02 14:22:40 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...


2026/06/02 14:22:40 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


🏃 View run hello-system-metrics at: http://127.0.0.1:5001/#/experiments/7/runs/71b01f7237834f0b89cf30a6b63f5e89
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/7
13 system/* metrics logged:

   system/cpu_utilization_percentage
   system/disk_available_megabytes
   system/disk_usage_megabytes
   system/disk_usage_percentage
   system/gpu_0_memory_usage_megabytes
   system/gpu_0_memory_usage_percentage
   system/gpu_0_power_usage_percentage
   system/gpu_0_power_usage_watts
   system/gpu_0_utilization_percentage
   system/network_receive_megabytes
   system/network_transmit_megabytes
   system/system_memory_usage_megabytes
   system/system_memory_usage_percentage


Those keys group into the four resource families plus GPU:

- **CPU** — `system/cpu_utilization_percentage`
- **Memory** — `system/system_memory_usage_megabytes`, `..._percentage`
- **Disk** — `system/disk_usage_megabytes`, `..._available_megabytes`, `..._percentage`
- **Network** — `system/network_receive_megabytes`, `system/network_transmit_megabytes`
- **GPU (per device `gpu_0`, `gpu_1`, …)** — `system/gpu_0_utilization_percentage`, `..._memory_usage_megabytes`, `..._memory_usage_percentage`, `..._power_usage_watts`, `..._power_usage_percentage`

If the `gpu_*` keys are **missing** from your output, `nvidia-ml-py` isn't installed (or there's no NVIDIA GPU) — MLflow logs the rest and moves on silently.

**Important: these are *host-wide*, not per-process.** `psutil` and NVML report the whole machine, so other processes (your browser, the MLflow server) count too. Read the charts as the machine's activity during the run, not your Python process in isolation.

## Step 2: A workload big enough to move the needle

California housing (~20 k rows, the repo's spine dataset) won't dent a modern workstation's RAM or touch the GPU — and you can't sample a tiny set *up*. So for the resource story we generate a **large synthetic regression set** with `make_regression`. Size it so the memory series climbs by a visible number of megabytes; it's tunable to your RAM.

In [3]:
N_SAMPLES = 5_000_000
N_FEATURES = 50

t = time.time()
X, y = make_regression(n_samples=N_SAMPLES, n_features=N_FEATURES, noise=0.1, random_state=0)
X = X.astype(np.float32)
y = y.astype(np.float32)
gb = X.nbytes / 1e9
print(f"generated X={X.shape} ({gb:.2f} GB float32) in {time.time() - t:.1f}s")

generated X=(5000000, 50) (1.00 GB float32) in 6.8s


## Step 3: CPU run — same algorithm, on the processor

We train XGBoost with `device="cpu"`. Watch two series: `system/cpu_utilization_percentage` should ride high (XGBoost uses all cores), and `system/system_memory_usage_megabytes` steps up while the `DMatrix` and model are resident.

In [4]:
params_cpu = {"device": "cpu", "tree_method": "hist", "objective": "reg:squarederror", "max_depth": 8}

with mlflow.start_run(run_name="train-cpu", log_system_metrics=True) as run_cpu:
    mlflow.log_params(params_cpu)
    t = time.time()
    dtrain = xgb.QuantileDMatrix(X, label=y)
    xgb.train(params_cpu, dtrain, num_boost_round=120)
    train_s = time.time() - t
    mlflow.log_metric("train_seconds", train_s)
    time.sleep(3)  # let the sampler capture the tail
    run_cpu_id = run_cpu.info.run_id

print(f"CPU train: {train_s:.1f}s  (run {run_cpu_id[:8]})")

2026/06/02 14:22:47 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


2026/06/02 14:23:42 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...


2026/06/02 14:23:42 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


🏃 View run train-cpu at: http://127.0.0.1:5001/#/experiments/7/runs/643d4073dd0d4e2db7d76267ea38fc99
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/7
CPU train: 52.5s  (run 643d4073)


## Step 4: GPU run — the *same* data, offloaded to CUDA

Now flip a single argument, `device="cuda"`. XGBoost moves the histogram construction onto the GPU. Expect `system/gpu_0_utilization_percentage` and `system/gpu_0_memory_usage_megabytes` to come alive, while CPU utilization drops relative to the CPU run.

(Aside: scikit-learn's `RandomForestRegressor` is **CPU-only** — it has no `device` argument and will never light up the GPU. XGBoost is used here precisely because the *same* learner runs on either device, making the contrast apples-to-apples.)

In [5]:
params_gpu = {**params_cpu, "device": "cuda"}

with mlflow.start_run(run_name="train-gpu", log_system_metrics=True) as run_gpu:
    mlflow.log_params(params_gpu)
    t = time.time()
    dtrain_gpu = xgb.QuantileDMatrix(X, label=y)
    xgb.train(params_gpu, dtrain_gpu, num_boost_round=120)
    train_s = time.time() - t
    mlflow.log_metric("train_seconds", train_s)
    time.sleep(3)
    run_gpu_id = run_gpu.info.run_id

print(f"GPU train: {train_s:.1f}s  (run {run_gpu_id[:8]})")

2026/06/02 14:23:42 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


2026/06/02 14:23:57 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...


2026/06/02 14:23:57 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


🏃 View run train-gpu at: http://127.0.0.1:5001/#/experiments/7/runs/ad9bb820b259428a8246bbeedf41561a
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/7
GPU train: 11.7s  (run ad9bb820)


## Step 5: Read the two runs side by side

Peak values straight from the logged history. The point isn't the exact numbers (host-wide, machine-specific) — it's the **contrast**: where each run spent its resources.

In [6]:
def peak(run_id, key):
    h = client.get_metric_history(run_id, key)
    return max((p.value for p in h), default=float("nan")), len(h)


def summarize(run_id, label):
    print(f"--- {label} ({run_id[:8]}) ---")
    for key in [
        "system/cpu_utilization_percentage",
        "system/system_memory_usage_megabytes",
        "system/gpu_0_utilization_percentage",
        "system/gpu_0_memory_usage_megabytes",
    ]:
        p, n = peak(run_id, key)
        print(f"  peak {key.replace('system/', ''):42} {p:8.1f}   ({n} samples)")


summarize(run_cpu_id, "CPU run")
summarize(run_gpu_id, "GPU run")

--- CPU run (643d4073) ---
  peak cpu_utilization_percentage                     95.4   (50 samples)


  peak system_memory_usage_megabytes               18227.0   (50 samples)
  peak gpu_0_utilization_percentage                   48.0   (50 samples)
  peak gpu_0_memory_usage_megabytes                 1344.3   (50 samples)
--- GPU run (ad9bb820) ---


  peak cpu_utilization_percentage                     76.1   (13 samples)
  peak system_memory_usage_megabytes               18628.7   (13 samples)


  peak gpu_0_utilization_percentage                   99.0   (13 samples)
  peak gpu_0_memory_usage_megabytes                 1917.6   (13 samples)


Two things to read off the numbers above.

**Speed.** On this machine the GPU run trained the *same* 120 rounds on the *same* 5 M × 50 data in roughly **a quarter of the wall-clock time** of the CPU run (≈12 s vs ≈52 s) — the headline reason you'd reach for `device="cuda"` at all.

**Where the work went.** The GPU run pushes `gpu_0_utilization_percentage` to ~99% and lifts `gpu_0_memory_usage_megabytes`; the CPU run leans harder on `cpu_utilization_percentage` (~95% vs ~76%) instead.

Notice the CPU run's GPU readings are **not zero** (here ~48% util, ~1.3 GB) even though XGBoost ran on the CPU. That's not a bug — it's the **host-wide** caveat from Step 1 made concrete: NVML reports the whole GPU (desktop compositor, other processes, this notebook's own earlier GPU work), not your one training call. So read the contrast as *relative* — GPU run ~99% vs CPU run ~48% — not as absolute per-process attribution. That single-flag difference, made visible, is the whole teaching point: **the charts let you confirm the GPU is actually doing the work you're paying it for.**

## Step 6: Viewing it in the UI

Open <http://127.0.0.1:5001/> → **System Metrics**. Click a training run and find the **System metrics** charts (alongside the model metrics):

- Set the x-axis to **wall-clock time** — system metrics are inherently time-based (unlike the *step*-based curves in `e_`).
- Compare `train-cpu` and `train-gpu`: the CPU panel and GPU panel swap prominence between them.
- `system_memory_usage_megabytes` shows the absolute footprint; on a machine with tens of GB of RAM, a few-GB array is a small *percentage* but a clearly visible step in *megabytes* — which is why the MB series is the one to watch when right-sizing.

**Teaching point — why production teams log this at all.** These charts are how you decide "this job needs a GPU / doesn't," "we're over-provisioned on RAM," or "the bottleneck is I/O, not compute." That's cost and capacity planning, grounded in evidence instead of vibes.

## Caveats

- **No GPU series without `nvidia-ml-py`.** Missing bindings → `system/gpu_*` is silently skipped, everything else still logs.
- **Host-wide, not per-process.** Other processes inflate the numbers; treat them as machine-level.
- **Short runs, few points.** The sampler needs at least one interval; sub-second runs may log almost nothing. Drop the interval (as we did) and/or `time.sleep` briefly so the tail is captured.
- **Sampling has a small cost.** A 1 s interval is fine; don't sample at, say, 10 ms in a tight production loop.

## Next steps

- **Enable it everywhere:** `mlflow.enable_system_metrics_logging()` once, or set `MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING=true` — no per-run flag needed.
- **The capstone (`k_`)** turns this on for its heavy tuning step, so the lifecycle run carries a resource story next to the accuracy story.
- **[System metrics docs](https://mlflow.org/docs/latest/ml/tracking/system-metrics/)** — the full metric list and configuration, including custom collectors.